# Practical 1A — MNIST MLP Backpropagation

## Aim
Implement backpropagation for MNIST using MLPs with one, two and three hidden layers and a softmax output.

## Setup
Run the installation cell if required. These notebooks use Python 3.10–3.12 where possible. On Apple Silicon, TensorFlow/PyTorch acceleration depends on the installed build; CPU execution is supported.

**Important:** dataset downloads and model training may take time. Results are generated when you run the notebook; no accuracy values are pre-filled.

In [ ]:
%pip install -q numpy pandas matplotlib scikit-learn tensorflow

## Reproducibility\nA fixed random seed is used where practical. If a dataset download fails, check your internet connection and rerun the loading cell.

## 1. Import libraries and load MNIST

In [ ]:
import time, numpy as np, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
np.random.seed(42); tf.random.set_seed(42)
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
x_train = x_train.reshape(-1, 784); x_test = x_test.reshape(-1, 784)
print(x_train.shape, x_test.shape)

## 2. Helper functions

In [ ]:
def plot_history(history, title="Training curves"):
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(history.history["loss"], label="train")
    if "val_loss" in history.history: ax[0].plot(history.history["val_loss"], label="validation")
    ax[0].set_title("Loss"); ax[0].legend()
    ax[1].plot(history.history["accuracy"], label="train")
    if "val_accuracy" in history.history: ax[1].plot(history.history["val_accuracy"], label="validation")
    ax[1].set_title("Accuracy"); ax[1].legend()
    fig.suptitle(title); plt.show()

## 3. Build and train MLPs with one, two and three hidden layers\nThe output uses softmax for the ten digit classes. Keras automatically computes gradients by backpropagation.

In [ ]:
def build_mlp(hidden_layers):
    model = keras.Sequential([layers.Input(shape=(784,))])
    for units in hidden_layers:
        model.add(layers.Dense(units, activation="relu"))
    model.add(layers.Dense(10, activation="softmax"))
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

architectures = {"1 hidden layer":[128], "2 hidden layers":[256,128], "3 hidden layers":[256,128,64]}
results = {}
for name, widths in architectures.items():
    model = build_mlp(widths)
    start = time.time()
    hist = model.fit(x_train, y_train, epochs=5, batch_size=128, validation_split=0.1, verbose=1)
    elapsed = time.time()-start
    loss, acc = model.evaluate(x_test, y_test, verbose=0)
    results[name] = {"model":model, "history":hist, "time":elapsed, "loss":loss, "accuracy":acc}
    print(name, "test accuracy:", round(acc,4), "test loss:", round(loss,4), "seconds:", round(elapsed,1))

In [ ]:
for name, r in results.items():
    plot_history(r["history"], name)

## 4. Comparison and conclusion

In [ ]:
import pandas as pd
comparison = pd.DataFrame([{ "Architecture":k, "Test accuracy":v["accuracy"], "Test loss":v["loss"], "Training time (s)":v["time"]} for k,v in results.items()])
display(comparison)

**Conclusion:** Compare the measured test accuracy, loss curves and training time. More hidden layers increase representational capacity but do not guarantee better test performance.